# Constant-Velocity Baseline

This notebook evaluates a simple hand-written maneuver baseline on the exact same split used by the learned models.


## Run Safety

Before running this notebook, restart the kernel.

Why this matters:
- previous model runs can leave tensors and cached arrays in memory
- old variables can leak into the current run and make comparisons unfair
- we want every notebook to save a clean, reproducible result

Recommended workflow:
1. Restart kernel
2. Run all cells from top to bottom
3. Let the notebook save its outputs before closing it


In [1]:
# Memory cleanup before starting this notebook.
import gc

gc.collect()
try:
    import torch
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
except Exception:
    pass

print('Kernel memory cleanup complete. Start the notebook from here after a restart.')


Kernel memory cleanup complete. Start the notebook from here after a restart.


In [2]:
# Standard-library imports used throughout the evaluation notebooks.
import json
import math
import random
import sys
from collections import Counter
from dataclasses import dataclass
from functools import lru_cache
from pathlib import Path
from datetime import datetime
from IPython.display import display

# Core scientific / ML stack.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset, Subset

# ROC / PR curves are useful for classification comparison if sklearn is available.
try:
    from sklearn.metrics import roc_curve, auc, precision_recall_curve, average_precision_score
    from sklearn.preprocessing import label_binarize
    SKLEARN_AVAILABLE = True
except Exception:
    SKLEARN_AVAILABLE = False

# Resolve the project root robustly no matter where Jupyter was launched from.
SEARCH_ROOT = Path.cwd().resolve()
PROJECT_ROOT = SEARCH_ROOT
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / '04_model_evaluation').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

EVAL_ROOT = PROJECT_ROOT / '04_model_evaluation'
THESIS_ROOT = EVAL_ROOT
NOTEBOOK_ROOT = EVAL_ROOT / 'notebooks'
RESULTS_ROOT = EVAL_ROOT / 'results'
WEIGHTS_ROOT = EVAL_ROOT / 'model_weights'
SPLITS_ROOT = EVAL_ROOT / 'splits'
PLOTS_ROOT = EVAL_ROOT / 'plots'
if str(NOTEBOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_ROOT))
ORIGINAL_DATASET_ROOT = PROJECT_ROOT / '03_dataset' / 'hybrid_maneuver_dataset'
PREFERRED_DATASET_ROOT = Path('/media/basudeo/1044063744061FD8/hybrid_maneuver_dataset')
DATASET_ROOT = PREFERRED_DATASET_ROOT if PREFERRED_DATASET_ROOT.exists() else ORIGINAL_DATASET_ROOT

print('Project root:', PROJECT_ROOT)
print('Evaluation workspace:', THESIS_ROOT)
print('Original dataset root:', ORIGINAL_DATASET_ROOT)
print('Preferred external dataset root:', PREFERRED_DATASET_ROOT)
print('Dataset root:', DATASET_ROOT)
print('Torch version:', torch.__version__)
print('Sklearn available for ROC/PR plots:', SKLEARN_AVAILABLE)


Project root: /home/basudeo/Documents/Thesis
Thesis workspace: /home/basudeo/Documents/Thesis/Thesis_Repo
Original dataset root: /home/basudeo/Documents/Thesis/hybrid_maneuver_dataset
Preferred external dataset root: /media/basudeo/1044063744061FD8/hybrid_maneuver_dataset
Dataset root: /media/basudeo/1044063744061FD8/hybrid_maneuver_dataset
Torch version: 2.3.1+cu118
Sklearn available for ROC/PR plots: True


In [3]:
# Shared experiment configuration.
# Keep this block explicit and heavily commented because it is the main place
# you will tune once more data has been collected.
LABEL_MODE = 'reduced'      # Use the balanced 5-class setting by default.
SEED = 42                   # Fix the split and initialization for reproducibility.
PAST_LEN = 10               # Number of past timesteps used to predict the current maneuver.
FUTURE_LEN = 5             # Number of future steps used for trajectory targets (ADE/FDE/RMSE).
SCAN_BEAMS = 512            # Number of lidar beams after resampling.
RANGE_CLIP = 30.0           # Maximum lidar range used for normalization.
BATCH_SIZE = 32             # Batch size for trainable models.
EPOCHS = 30                 # Upper bound; early stopping will usually stop sooner.
EARLY_STOPPING_PATIENCE = 5 # Patience on validation macro-F1.
LR = 1e-3                   # Adam learning rate.
WEIGHT_DECAY = 1e-4         # Small regularization on trainable models.
TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
MAX_SAMPLES = None          # Set to an integer for quick smoke tests.
USE_CPU = False             # Force CPU if needed.

# Hidden sizes for the current classification baselines.
CNN_HIDDEN = 96
GRAPH_HIDDEN = 96
FUSION_HIDDEN = 128
LSTM_HIDDEN = 128
LSTM_LAYERS = 1
MSG_PASSES = 2
DROPOUT = 0.10
TRANSFORMER_HEADS = 4
TRANSFORMER_LAYERS = 2
TRANSFORMER_FF = 256

# Device selection is kept simple and transparent.
device = torch.device('cuda' if torch.cuda.is_available() and not USE_CPU else 'cpu')
print('Device:', device)


Device: cuda


In [4]:
# Shared data and evaluation utilities live in one helper module so that
# all notebooks reuse the same dataset, split, path-remapping, and metric logic.
from dataset_helper import (
    SKLEARN_AVAILABLE,
    build_class_weights,
    build_label_mapping,
    build_run_manifest,
    build_sample_table,
    canonical_agent_order,
    compute_classification_metrics_from_probs,
    compute_trajectory_metrics,
    configure_helper,
    discover_frame_files,
    edge_features_for_order,
    group_streams,
    load_npy_cached,
    node_feature,
    prepare_result_dirs,
    remap_dataset_path,
    resample_scan,
    save_confusion_matrix,
    save_history_plot,
    save_mean_step_error_plot,
    save_or_load_fixed_split,
    save_predictions_csv,
    save_roc_pr_curves,
    save_run_manifest,
    save_training_history,
    save_trajectory_overlay_plots,
    set_seed,
    timestamp_tag,
)

configure_helper(
    dataset_root=DATASET_ROOT,
    original_dataset_root=ORIGINAL_DATASET_ROOT,
    results_root=RESULTS_ROOT,
    weights_root=WEIGHTS_ROOT,
)


In [5]:
@dataclass
class ClassificationSample:
    scan_seq: torch.Tensor | None
    node_seq: torch.Tensor | None
    edge_seq: torch.Tensor | None
    label: torch.Tensor
    future_xy: torch.Tensor
    future_dt: torch.Tensor
    sample_id: str


class BaseClassificationDataset(Dataset):
    """Base dataset that turns JSONL streams into time windows.

    Subclasses decide which modalities they need from each window.
    """
    def __init__(self, streams, sample_table, label_to_idx, past_len, scan_beams, range_clip):
        self.streams = streams
        self.sample_table = sample_table
        self.label_to_idx = label_to_idx
        self.past_len = past_len
        self.scan_beams = scan_beams
        self.range_clip = range_clip

    def __len__(self):
        return len(self.sample_table)

    def _window(self, idx):
        meta = self.sample_table[idx]
        stream = self.streams[meta['stream_idx']]
        return stream[meta['start']: meta['start'] + self.past_len], meta


class ScanOnlyDataset(BaseClassificationDataset):
    def __getitem__(self, idx):
        window, meta = self._window(idx)
        scan_seq = []
        for frame in window:
            scan_ref = frame['modalities']['ego_planar_scan']
            scan = load_npy_cached(str(scan_ref['path']))
            scan_seq.append(resample_scan(scan, self.scan_beams, self.range_clip))
        label_idx = self.label_to_idx[meta['label']]
        return ClassificationSample(
            scan_seq=torch.tensor(np.stack(scan_seq, axis=0), dtype=torch.float32),
            node_seq=None,
            edge_seq=None,
            label=torch.tensor(label_idx, dtype=torch.long),
            future_xy=torch.tensor(meta['future_xy'], dtype=torch.float32),
            future_dt=torch.tensor(meta['future_dt'], dtype=torch.float32),
            sample_id=meta['sample_id'],
        )


class GraphOnlyDataset(BaseClassificationDataset):
    def __getitem__(self, idx):
        window, meta = self._window(idx)
        node_seq, edge_seq = [], []
        ego_id = meta['ego_id']
        order = canonical_agent_order(ego_id)
        for frame in window:
            agents = frame['agents']
            ego_state = agents[ego_id]['state']
            node_seq.append([node_feature(agents[agent_id], ego_state) for agent_id in order])
            edge_seq.append(edge_features_for_order(frame, order))
        label_idx = self.label_to_idx[meta['label']]
        return ClassificationSample(
            scan_seq=None,
            node_seq=torch.tensor(node_seq, dtype=torch.float32),
            edge_seq=torch.tensor(edge_seq, dtype=torch.float32),
            label=torch.tensor(label_idx, dtype=torch.long),
            future_xy=torch.tensor(meta['future_xy'], dtype=torch.float32),
            future_dt=torch.tensor(meta['future_dt'], dtype=torch.float32),
            sample_id=meta['sample_id'],
        )


class ScanGraphDataset(BaseClassificationDataset):
    def __getitem__(self, idx):
        window, meta = self._window(idx)
        scan_seq, node_seq, edge_seq = [], [], []
        ego_id = meta['ego_id']
        order = canonical_agent_order(ego_id)
        for frame in window:
            scan_ref = frame['modalities']['ego_planar_scan']
            scan = load_npy_cached(str(scan_ref['path']))
            scan_seq.append(resample_scan(scan, self.scan_beams, self.range_clip))
            agents = frame['agents']
            ego_state = agents[ego_id]['state']
            node_seq.append([node_feature(agents[agent_id], ego_state) for agent_id in order])
            edge_seq.append(edge_features_for_order(frame, order))
        label_idx = self.label_to_idx[meta['label']]
        return ClassificationSample(
            scan_seq=torch.tensor(np.stack(scan_seq, axis=0), dtype=torch.float32),
            node_seq=torch.tensor(node_seq, dtype=torch.float32),
            edge_seq=torch.tensor(edge_seq, dtype=torch.float32),
            label=torch.tensor(label_idx, dtype=torch.long),
            future_xy=torch.tensor(meta['future_xy'], dtype=torch.float32),
            future_dt=torch.tensor(meta['future_dt'], dtype=torch.float32),
            sample_id=meta['sample_id'],
        )


def collate_samples(batch):
    scan_seq = None if batch[0].scan_seq is None else torch.stack([sample.scan_seq for sample in batch], dim=0)
    node_seq = None if batch[0].node_seq is None else torch.stack([sample.node_seq for sample in batch], dim=0)
    edge_seq = None if batch[0].edge_seq is None else torch.stack([sample.edge_seq for sample in batch], dim=0)
    labels = torch.stack([sample.label for sample in batch], dim=0)
    future_xy = torch.stack([sample.future_xy for sample in batch], dim=0)
    future_dt = torch.stack([sample.future_dt for sample in batch], dim=0)
    sample_ids = [sample.sample_id for sample in batch]
    return scan_seq, node_seq, edge_seq, labels, future_xy, future_dt, sample_ids


In [6]:
set_seed(SEED)
labels, label_mapping = build_label_mapping(LABEL_MODE)
label_to_idx = {label: idx for idx, label in enumerate(labels)}
streams = group_streams(DATASET_ROOT, allowed_labels=set(labels), label_mapping=label_mapping)
sample_table = build_sample_table(streams, past_len=PAST_LEN, future_len=FUTURE_LEN)
split_path = SPLITS_ROOT / f'classification_split_{LABEL_MODE}_seed{SEED}_past{PAST_LEN}_future{FUTURE_LEN}.json'
split_info = save_or_load_fixed_split(sample_table, split_path, SEED, TRAIN_RATIO, VAL_RATIO, PAST_LEN, FUTURE_LEN)

print('Labels:', labels)
print('Split path:', split_path)
print('Total samples in canonical table:', len(sample_table))
print('Train / Val / Test:', len(split_info['train_indices']), len(split_info['val_indices']), len(split_info['test_indices']))
print('Future horizon:', split_info['future_len'])


Labels: ['go_to_goal', 'avoid_left', 'avoid_right', 'commit_forward', 'arrived']
Split path: /home/basudeo/Documents/Thesis/Thesis_Repo/splits/classification_split_reduced_seed42_past10_future5.json
Total samples in canonical table: 11488
Train / Val / Test: 2844 609 610
Future horizon: 5


In [7]:
MODEL_SLUG = 'cv_baseline'
TIMESTAMP = timestamp_tag()
result_dir, weight_dir, plot_dir = prepare_result_dirs(MODEL_SLUG)

# The constant-velocity baseline is deliberately simple.
# It uses the final step in each temporal window and converts command + front scan
# context into a maneuver guess. This gives us a cheap reference point.
dataset = ScanGraphDataset(
    streams=streams,
    sample_table=sample_table,
    label_to_idx=label_to_idx,
    past_len=PAST_LEN,
    scan_beams=SCAN_BEAMS,
    range_clip=RANGE_CLIP,
)

test_subset = Subset(dataset, split_info['test_indices'])
test_loader = DataLoader(test_subset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_samples)

CV_GOAL_TOLERANCE = 1.5
CV_ANGULAR_THRESHOLD = 0.35
CV_COMMIT_LINEAR_THRESHOLD = 0.5
CV_FRONT_WINDOW = 32
CV_FRONT_CLEARANCE_METERS = 4.0

all_probs, all_targets, all_sample_ids = [], [], []
all_pred_future_xy, all_true_future_xy = [], []
for scan_seq, node_seq, edge_seq, labels_batch, future_xy_batch, future_dt_batch, sample_ids in test_loader:
    last_scan = scan_seq[:, -1]
    ego_last = node_seq[:, -1, 0]

    goal_dist = torch.sqrt(ego_last[:, 7] ** 2 + ego_last[:, 8] ** 2 + ego_last[:, 9] ** 2)
    linear_cmd = ego_last[:, 10]
    angular_cmd = ego_last[:, 11]

    center = last_scan.shape[-1] // 2
    front_slice = last_scan[:, 0, max(0, center - CV_FRONT_WINDOW): min(last_scan.shape[-1], center + CV_FRONT_WINDOW)]
    front_clearance = front_slice.min(dim=1).values * RANGE_CLIP

    pred_idx = torch.full_like(labels_batch, fill_value=label_to_idx['go_to_goal'])
    if 'arrived' in label_to_idx:
        pred_idx = torch.where(goal_dist <= CV_GOAL_TOLERANCE, torch.full_like(pred_idx, label_to_idx['arrived']), pred_idx)
    if 'avoid_left' in label_to_idx:
        pred_idx = torch.where((goal_dist > CV_GOAL_TOLERANCE) & (angular_cmd >= CV_ANGULAR_THRESHOLD), torch.full_like(pred_idx, label_to_idx['avoid_left']), pred_idx)
    if 'avoid_right' in label_to_idx:
        pred_idx = torch.where((goal_dist > CV_GOAL_TOLERANCE) & (angular_cmd <= -CV_ANGULAR_THRESHOLD), torch.full_like(pred_idx, label_to_idx['avoid_right']), pred_idx)
    if 'commit_forward' in label_to_idx:
        commit_mask = (
            (goal_dist > CV_GOAL_TOLERANCE)
            & (linear_cmd >= CV_COMMIT_LINEAR_THRESHOLD)
            & (torch.abs(angular_cmd) < CV_ANGULAR_THRESHOLD)
            & (front_clearance <= CV_FRONT_CLEARANCE_METERS)
        )
        pred_idx = torch.where(commit_mask, torch.full_like(pred_idx, label_to_idx['commit_forward']), pred_idx)

    # Constant-velocity trajectory baseline in ego-relative world coordinates.
    # We use the final observed ego velocity and extrapolate over the saved future timestamps.
    vx_last = ego_last[:, 3].unsqueeze(1)
    vy_last = ego_last[:, 4].unsqueeze(1)
    pred_future_xy = torch.stack([vx_last * future_dt_batch, vy_last * future_dt_batch], dim=-1)

    probs = torch.nn.functional.one_hot(pred_idx, num_classes=len(labels)).float().numpy()
    all_probs.append(probs)
    all_targets.append(labels_batch.numpy())
    all_pred_future_xy.append(pred_future_xy.numpy())
    all_true_future_xy.append(future_xy_batch.numpy())
    all_sample_ids.extend(sample_ids)

probabilities = np.concatenate(all_probs, axis=0)
targets = np.concatenate(all_targets, axis=0)
pred_future_xy = np.concatenate(all_pred_future_xy, axis=0)
true_future_xy = np.concatenate(all_true_future_xy, axis=0)
metrics, preds, confusion = compute_classification_metrics_from_probs(probabilities, targets, labels)
metrics.update(compute_trajectory_metrics(pred_future_xy, true_future_xy))
roc_summary = save_roc_pr_curves(probabilities, targets, labels, plot_dir)
metrics.update(roc_summary)
run_manifest = build_run_manifest(
    model_slug=MODEL_SLUG,
    timestamp=TIMESTAMP,
    labels=labels,
    split_path=split_path,
    extra={
        'baseline_type': 'constant_velocity_heuristic',
        'goal_tolerance': CV_GOAL_TOLERANCE,
        'angular_threshold': CV_ANGULAR_THRESHOLD,
        'commit_linear_threshold': CV_COMMIT_LINEAR_THRESHOLD,
        'front_window': CV_FRONT_WINDOW,
        'front_clearance_meters': CV_FRONT_CLEARANCE_METERS,
        'future_len': FUTURE_LEN,
        'trajectory_baseline': 'last_observed_velocity_world_frame',
    },
)
save_run_manifest(result_dir, run_manifest, TIMESTAMP)
metrics['split_path'] = str(split_path)

metrics_path = result_dir / f'metrics_{TIMESTAMP}.json'
metrics_path.write_text(json.dumps(metrics, indent=2))
(result_dir / 'latest_metrics.json').write_text(json.dumps(metrics, indent=2))
save_predictions_csv(all_sample_ids, targets, preds, probabilities, labels, result_dir / f'predictions_{TIMESTAMP}.csv')
save_confusion_matrix(confusion, labels, result_dir / f'confusion_{TIMESTAMP}.csv', plot_dir / f'confusion_{TIMESTAMP}.png', 'CV Baseline Confusion Matrix')

print('CV baseline metrics:')
print(json.dumps(metrics, indent=2))
display(pd.DataFrame(confusion, index=labels, columns=labels))


CV baseline metrics:
{
  "accuracy": 0.4163934426229508,
  "macro_precision": 0.2985635623437232,
  "macro_recall": 0.5410345874993978,
  "macro_f1": 0.316312934691758,
  "confusion_matrix": [
    [
      182,
      23,
      40,
      16,
      0
    ],
    [
      0,
      30,
      0,
      1,
      0
    ],
    [
      0,
      1,
      29,
      0,
      0
    ],
    [
      80,
      45,
      39,
      13,
      0
    ],
    [
      111,
      0,
      0,
      0,
      0
    ]
  ],
  "ADE": 1.5617413520812988,
  "FDE": 2.6017253398895264,
  "RMSE": 2.1821932792663574,
  "roc_auc_macro": 0.686325614954965,
  "pr_auc_macro": 0.3017005305331036,
  "status": "saved",
  "split_path": "/home/basudeo/Documents/Thesis/Thesis_Repo/splits/classification_split_reduced_seed42_past10_future5.json"
}


,go_to_goal,avoid_left,avoid_right,commit_forward,arrived
go_to_goal,182,23,40,16,0
avoid_left,0,30,0,1,0
avoid_right,0,1,29,0,0
commit_forward,80,45,39,13,0
arrived,111,0,0,0,0
